# Data-parallel SGD step with uneven shards

In data parallelism every rank holds a full replica of the model, computes gradients on its own slice of the batch, and an all-reduce combines them so that every replica takes the **same** optimizer step. The step must equal the one a single process would take on the whole batch, even when the last batch of an epoch leaves ranks with different numbers of samples, or none.

Simulate it in one process with PyTorch: one parameter replica per rank, autograd per rank, a summed "all-reduce", and an SGD update on every replica.

Write `dp_sgd_step(X, y, counts, W, b, lr)`.

## Inputs
- `X`: `float64` `(N, D)`; `y`: `int64` `(N,)` class labels in $[0, C)$.
- `counts`: list of $R \ge 1$ non-negative ints summing to $N \ge 1$: rank $r$ owns the next `counts[r]` rows of the batch, in order.
- `W`: `float64` `(D, C)`, `b`: `float64` `(C,)`: the shared initial parameters. `lr`: float.

## Model and loss
Logits $= XW + b$; the batch loss is the **mean** cross-entropy over all $N$ samples.

## Per step
1. Each rank $r$ with $n_r > 0$ makes its own leaf copies of `W` and `b`, computes the mean cross-entropy over its $n_r$ samples, and backpropagates to get local gradients $g_r$. A rank with $n_r = 0$ contributes zero gradients (and must not produce NaN).
2. All-reduce: the global gradient is $g = \dfrac{1}{N}\sum_r n_r\, g_r$.
3. Every replica applies $\theta \leftarrow \theta - \text{lr}\cdot g$.

## Output
A tuple `(W_all, b_all, flat_grad)`:
- `W_all`: `(R, D, C)` and `b_all`: `(R, C)`, the updated parameters of every replica (they should all be identical);
- `flat_grad`: `(D C + C,)`, the global gradient as one flat bucket: `W`'s gradient flattened in row-major order, followed by `b`'s.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import torch.nn.functional as F

def dp_sgd_step(X, y, counts, W, b, lr):
    """Return (W_all (R, D, C), b_all (R, C), flat_grad (D*C + C,))."""
    grads_W, grads_b, start = [], [], 0
    for n in counts:
        Wr = W.clone().requires_grad_(True)
        br = b.clone().requires_grad_(True)
        F.cross_entropy(X[start:start + n] @ Wr + br, y[start:start + n]).backward()
        grads_W.append(Wr.grad); grads_b.append(br.grad)
        start += n
    gW = sum(grads_W) / len(counts)              # TODO: equal weights per rank; NaN for empty ranks
    gb = sum(grads_b) / len(counts)
    R = len(counts)
    return (W - lr * gW).expand(R, *W.shape), (b - lr * gb).expand(R, *b.shape), torch.cat([gW.flatten(), gb])


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0, 1], "dtype": "int64"}, [1, 1], {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__nd__": [[[0.25, -0.25], [-0.25, 0.25]], [[0.25, -0.25], [-0.25, 0.25]]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [-0.25, 0.25, 0.25, -0.25, 0.0, 0.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [0, 1, 1], "dtype": "int64"}, [2, 1], {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__nd__": [[[0.0, 0.0], [-0.16666666666666666, 0.16666666666666666]], [[0.0, 0.0], [-0.16666666666666666, 0.16666666666666666]]], "dtype": "float64"}, {"__nd__": [[-0.08333333333333333, 0.08333333333333333], [-0.08333333333333333, 0.08333333333333333]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.3333333333333333, -0.3333333333333333, 0.16666666666666666, -0.16666666666666666], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.6955197700381686, -0.9794741683587314, -1.5734903329477068], [-2.924970571840865, -0.35323216358269055, 1.2476063726111246], [0.03307262346929485, 0.5118440276808229, 1.0232382136634648], [-0.8821458480598934, 2.6522866971695453, -0.8769082522563802], [0.3735530621878692, 2.7395808181161874, -0.11425241215164042], [0.11429451370904192, -0.5118795445527522, 0.33452648360440757], [-2.1266836963811473, -0.6468087625992227, 1.694093202177908], [0.21132613280362944, -0.24039851728557995, 0.372928487050425], [0.4146068139746092, 0.15210326387147993, 0.5785878662409928], [-1.6595968004978632, 0.5912552798907551, -0.6735267717733201]], "dtype": "float64"}, {"__nd__": [3, 3, 0, 3, 3, 3, 3, 3, 3, 3], "dtype": "int64"}, [5, 5], {"__nd__": [[0.5063658778276171, -0.014299130618634812, 0.4265458578392969, -0.5575280334118149], [0.8097086697584963, -0.15053065247546177, 0.6826329199862453, 0.42702567238030986], [-0.5504725674402294, 0.16157724258176703, -0.27122516221428544, 0.2046822439403161]], "dtype": "float64"}, {"__nd__": [0.143247855108075, 0.0688670835263633, 0.10679358893282981, -0.11715695130178541], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__nd__": [[[0.5085497807359637, -0.0022215018073985394, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]], [[0.5085497807359637, -0.0022215018073985394, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]]], "dtype": "float64"}, {"__nd__": [[0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246], [0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246]], "dtype": "float64"}, {"__nd__": [-0.02183902908346588, -0.12077628811236271, -0.017369532438666856, 0.15998484963449544, 0.19692237986703023, -0.03491821233757002, 0.1503959560547519, -0.31240012358421215, -0.1752771298801332, 0.09517169547953261, -0.016353698927875138, 0.0964591333284757, 0.16093838627740273, 0.21636872371062502, 0.21679805424155382, -0.5941051642295816], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.6955197700381686, -0.9794741683587314, -1.5734903329477068], [-2.924970571840865, -0.35323216358269055, 1.2476063726111246], [0.03307262346929485, 0.5118440276808229, 1.0232382136634648], [-0.8821458480598934, 2.6522866971695453, -0.8769082522563802], [0.3735530621878692, 2.7395808181161874, -0.11425241215164042], [0.11429451370904192, -0.5118795445527522, 0.33452648360440757], [-2.1266836963811473, -0.6468087625992227, 1.694093202177908], [0.21132613280362944, -0.24039851728557995, 0.372928487050425], [0.4146068139746092, 0.15210326387147993, 0.5785878662409928], [-1.6595968004978632, 0.5912552798907551, -0.6735267717733201]], "dtype": "float64"}, {"__nd__": [3, 3, 0, 3, 3, 3, 3, 3, 3, 3], "dtype": "int64"}, [7, 2, 1], {"__nd__": [[0.5063658778276171, -0.014299130618634812, 0.4265458578392969, -0.5575280334118149], [0.8097086697584963, -0.15053065247546177, 0.6826329199862453, 0.42702567238030986], [-0.5504725674402294, 0.16157724258176703, -0.27122516221428544, 0.2046822439403161]], "dtype": "float64"}, {"__nd__": [0.143247855108075, 0.0688670835263633, 0.10679358893282981, -0.11715695130178541], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__nd__": [[[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]], [[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]], [[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]]], "dtype": "float64"}, {"__nd__": [[0.12715401648033473, 0.047230211155300794, 0.08511378350867443, -0.057746434878827246], [0.12715401648033473, 0.047230211155300794, 0.08511378350867443, -0.057746434878827246], [0.12715401648033473, 0.047230211155300794, 0.08511378350867443, -0.057746434878827246]], "dtype": "float64"}, {"__nd__": [-0.02183902908346588, -0.12077628811236274, -0.017369532438666856, 0.1599848496344954, 0.19692237986703026, -0.03491821233757002, 0.15039595605475187, -0.3124001235842121, -0.17527712988013316, 0.0951716954795326, -0.01635369892787513, 0.09645913332847565, 0.16093838627740276, 0.21636872371062496, 0.2167980542415538, -0.5941051642295816], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.6955197700381686, -0.9794741683587314, -1.5734903329477068], [-2.924970571840865, -0.35323216358269055, 1.2476063726111246], [0.03307262346929485, 0.5118440276808229, 1.0232382136634648], [-0.8821458480598934, 2.6522866971695453, -0.8769082522563802], [0.3735530621878692, 2.7395808181161874, -0.11425241215164042], [0.11429451370904192, -0.5118795445527522, 0.33452648360440757], [-2.1266836963811473, -0.6468087625992227, 1.694093202177908], [0.21132613280362944, -0.24039851728557995, 0.372928487050425], [0.4146068139746092, 0.15210326387147993, 0.5785878662409928], [-1.6595968004978632, 0.5912552798907551, -0.6735267717733201]], "dtype": "float64"}, {"__nd__": [3, 3, 0, 3, 3, 3, 3, 3, 3, 3], "dtype": "int64"}, [0, 10, 0], {"__nd__": [[0.5063658778276171, -0.014299130618634812, 0.4265458578392969, -0.5575280334118149], [0.8097086697584963, -0.15053065247546177, 0.6826329199862453, 0.42702567238030986], [-0.5504725674402294, 0.16157724258176703, -0.27122516221428544, 0.2046822439403161]], "dtype": "float64"}, {"__nd__": [0.143247855108075, 0.0688670835263633, 0.10679358893282981, -0.11715695130178541], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__nd__": [[[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]], [[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]], [[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]]], "dtype": "float64"}, {"__nd__": [[0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246], [0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246], [0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246]], "dtype": "float64"}, {"__nd__": [-0.021839029083465876, -0.12077628811236274, -0.017369532438666853, 0.15998484963449544, 0.19692237986703023, -0.03491821233757002, 0.15039595605475187, -0.3124001235842121, -0.1752771298801332, 0.09517169547953261, -0.016353698927875138, 0.09645913332847567, 0.16093838627740273, 0.21636872371062496, 0.21679805424155382, -0.5941051642295816], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.6955197700381686, -0.9794741683587314, -1.5734903329477068], [-2.924970571840865, -0.35323216358269055, 1.2476063726111246], [0.03307262346929485, 0.5118440276808229, 1.0232382136634648], [-0.8821458480598934, 2.6522866971695453, -0.8769082522563802], [0.3735530621878692, 2.7395808181161874, -0.11425241215164042], [0.11429451370904192, -0.5118795445527522, 0.33452648360440757], [-2.1266836963811473, -0.6468087625992227, 1.694093202177908], [0.21132613280362944, -0.24039851728557995, 0.372928487050425], [0.4146068139746092, 0.15210326387147993, 0.5785878662409928], [-1.6595968004978632, 0.5912552798907551, -0.6735267717733201]], "dtype": "float64"}, {"__nd__": [3, 3, 0, 3, 3, 3, 3, 3, 3, 3], "dtype": "int64"}, [10], {"__nd__": [[0.5063658778276171, -0.014299130618634812, 0.4265458578392969, -0.5575280334118149], [0.8097086697584963, -0.15053065247546177, 0.6826329199862453, 0.42702567238030986], [-0.5504725674402294, 0.16157724258176703, -0.27122516221428544, 0.2046822439403161]], "dtype": "float64"}, {"__nd__": [0.143247855108075, 0.0688670835263633, 0.10679358893282981, -0.11715695130178541], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__nd__": [[[0.5085497807359637, -0.0022215018073985376, 0.4282828110831636, -0.5735265183752645], [0.7900164317717933, -0.14703883124170478, 0.6675933243807701, 0.4582656847387311], [-0.5329448544522161, 0.15206007303381378, -0.26958979232149793, 0.19503633060746853]]], "dtype": "float64"}, {"__nd__": [[0.12715401648033473, 0.047230211155300794, 0.08511378350867442, -0.057746434878827246]], "dtype": "float64"}, {"__nd__": [-0.021839029083465876, -0.12077628811236274, -0.017369532438666853, 0.15998484963449544, 0.19692237986703023, -0.03491821233757002, 0.15039595605475187, -0.3124001235842121, -0.1752771298801332, 0.09517169547953261, -0.016353698927875138, 0.09645913332847567, 0.16093838627740273, 0.21636872371062496, 0.21679805424155382, -0.5941051642295816], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.20394519519921483, 0.3590717720756556, 1.0888977064622498, -0.6194233271487931, 0.5687860933561971], [0.42947130904497827, -1.524318371004195, -1.0397233608240872, 0.4446708098360002, -1.1765680129615403], [-1.226916373430967, 0.8952400173622894, 0.28556628961744895, -0.23515623857913578, 0.10549359303397024], [-0.40452582539599136, -0.5369294030017739, -0.4605884349903233, 0.38606190863397244, 1.3213036842667867], [-0.4707568598432348, -0.06865727792537184, -0.2182780982196439, -1.2600277603013605, 0.25194139300863316], [0.8667653893827483, -0.8436849831309196, 0.8330973617910787, -0.5716675976551412, 1.1513005043097746], [1.2158655860944216, -0.383072989490739, -2.0009402393427296, -0.3479884977165963, -1.3912077061519135], [-1.3147207307641426, -0.2528270425719888, 1.3604499417455416, -0.09210652557839855, -0.219580138267512], [-1.8504993469101834, -0.6061244605987609, -1.3950831707321905, 0.5203277449467695, -0.5282455885197661], [-0.04390937410231113, -0.016708428109463203, -0.18777482468081588, 0.9525678096640364, 0.4867924615107313], [-0.31962727095953336, 0.15375911656247143, 0.5527752284525819, 1.3713452319255217, 0.12982802652255646], [1.447618942170459, -0.5941755106420729, -0.23383170493291058, -1.4872048839382854, 0.8218959189256102], [-1.0326085772321911, -1.032463793574626, 0.8420940709449695, -0.5326586529478279, -0.7531147387143058], [0.9530456660312242, -0.18430028499434284, 0.9203384889851014, -0.40268257446727856, 0.7266658873734084], [1.0785411130489224, 0.08229186612716916, 1.7989992301457534, -0.39614053041389063, -0.21173031778943865], [0.0793709941333931, 1.005520494651668, -0.2875935809547487, -0.4365914827584129, 0.2537373606441843], [1.2498472087876555, -0.416565435932181, -1.277652251622201, -0.1308513963358957, 0.623763132298492], [-0.542791870640899, 0.12424713542015912, -1.8689870874381547, 0.021277643835543144, 0.10273796020804157], [0.6224986114518901, 0.3092743003403864, 0.7717776154755547, -0.9943756131385594, 0.6602886510183212], [-1.6512569638236714, -0.5968541812774073, 1.105857448839561, 0.851183379839743, -1.8530139843897926], [0.5884176623353239, -0.43080228767571965, -0.7779212006140349, 0.20445320686653481, 0.12890809713691617], [1.3943221727811774, -1.441975518609771, -0.4283761626487952, -0.14254520739113477, -1.8406313429052836], [1.1730940844306155, 1.756085887607632, -0.62511989710298, 0.3815532399707075, -0.26910628695698346]], "dtype": "float64"}, {"__nd__": [1, 1, 1, 0, 2, 2, 0, 2, 0, 1, 0, 2, 2, 2, 0, 0, 0, 0, 2, 1, 0, 0, 1], "dtype": "int64"}, [6, 6, 6, 5], {"__nd__": [[-0.1399241221606185, 0.43511420797926476, -0.9195142527052586], [-1.4350772441506012, -1.7337679288327537, 0.22085289280436593], [1.2766907986927296, 1.1718821391201832, 0.03375359913436464], [-0.9616653772653441, -0.36329692701267274, -1.507890710749208], [-1.5763462762873433, 1.0659444941812506, -0.27814878318401204]], "dtype": "float64"}, {"__nd__": [0.314441516736664, -0.27350567994765285, 1.3249204005210136], "dtype": "float64"}, 0.05], "e": {"__tup__": [{"__nd__": [[[-0.13491040027314288, 0.4258608628408945, -0.915274629454364], [-1.4308135991556041, -1.72797819540239, 0.21079951437900515], [1.259820738295756, 1.1707255783760178, 0.05178022027550361], [-0.9597426221215758, -0.35701772102758594, -1.5160926718780632], [-1.5676959142112468, 1.0534623742643132, -0.2743170253431712]], [[-0.13491040027314288, 0.4258608628408945, -0.915274629454364], [-1.4308135991556041, -1.72797819540239, 0.21079951437900515], [1.259820738295756, 1.1707255783760178, 0.05178022027550361], [-0.9597426221215758, -0.35701772102758594, -1.5160926718780632], [-1.5676959142112468, 1.0534623742643132, -0.2743170253431712]], [[-0.13491040027314288, 0.4258608628408945, -0.915274629454364], [-1.4308135991556041, -1.72797819540239, 0.21079951437900515], [1.259820738295756, 1.1707255783760178, 0.05178022027550361], [-0.9597426221215758, -0.35701772102758594, -1.5160926718780632], [-1.5676959142112468, 1.0534623742643132, -0.2743170253431712]], [[-0.13491040027314288, 0.4258608628408945, -0.915274629454364], [-1.4308135991556041, -1.72797819540239, 0.21079951437900515], [1.259820738295756, 1.1707255783760178, 0.05178022027550361], [-0.9597426221215758, -0.35701772102758594, -1.5160926718780632], [-1.5676959142112468, 1.0534623742643132, -0.2743170253431712]]], "dtype": "float64"}, {"__nd__": [[0.3214108992688377, -0.27244780545069275, 1.3168931434918798], [0.3214108992688377, -0.27244780545069275, 1.3168931434918798], [0.3214108992688377, -0.27244780545069275, 1.3168931434918798], [0.3214108992688377, -0.27244780545069275, 1.3168931434918798]], "dtype": "float64"}, {"__nd__": [-0.10027443774951254, 0.18506690276740437, -0.08479246501789177, -0.08527289989994326, -0.11579466860727221, 0.20106756850721552, 0.33740120793946954, 0.023131214883309787, -0.36053242282277936, -0.038455102875366555, -0.12558411970173575, 0.16403922257710232, -0.1730072415219315, 0.24964239833874813, -0.0766351568168166, -0.1393876506434746, -0.021157489939202287, 0.16054514058267685], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.4375255926601447, -0.7761577399792198, -0.3660937926131038, -0.6752188274196901], [1.3177607937198326, 1.2466016502499169, -0.2902878268806411, 2.299392727793735], [0.02640738413905812, -1.2159340754805035, -2.0333257583952613, -0.14375006981088342], [0.3213200068496095, -0.5866035100561903, 0.36309552910733295, 1.1950846827570762], [-0.05982354325972556, 0.8428941234726923, 1.6645750853020878, 0.4977159293553845], [-0.5805693055183667, 0.4318041081460906, 0.5366909933858184, 2.1374528604300718], [-0.4345689341851538, -1.5503831767944598, 0.2988020484784649, 0.3257941493500045], [-0.4156790739431062, 0.5647275930001924, 0.31822397113899203, 1.2110095061165949]], "dtype": "float64"}, {"__nd__": [5, 5, 5, 4, 2, 4, 0, 1], "dtype": "int64"}, [1, 0, 3, 0, 4], {"__nd__": [[-0.5077870477362139, 1.2710858941464, 0.5791211203916508, -0.022218595716001575, 1.527722135359455, 0.5015889572318717], [0.9994900677226612, 0.8050143701530288, -0.1474115407619588, -0.14031855353665593, 0.00735619364245958, -1.0909183334940347], [-0.3449326725783397, 0.8542757774328085, 0.3097283045736454, -0.9438029600845004, 2.363450803699267, -1.0301764872720272], [0.6225975377550774, 0.7253857327335095, 1.277256183371252, -0.8594165799580201, 0.024978281793473993, -0.19065129949129894]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__nd__": [[[-0.5409205894969416, 1.1302042956477851, 0.558862701468178, -0.04577449992525383, 1.4629958484374341, 0.7841447075459602], [0.7681347064675581, 0.7698865668464419, -0.056119575308986924, -0.07264464262556042, -0.039179482856477606, -0.9368653687974748], [-0.3413907263607456, 0.8510267256748743, 0.45388728871916123, -0.8873483320810232, 2.3061526859499213, -1.1737848761313343], [0.5116470948621722, 0.5885168468728869, 1.0119527734469473, -0.8531670963007332, 0.32886167790602977, 0.01233855941669082]], [[-0.5409205894969416, 1.1302042956477851, 0.558862701468178, -0.04577449992525383, 1.4629958484374341, 0.7841447075459602], [0.7681347064675581, 0.7698865668464419, -0.056119575308986924, -0.07264464262556042, -0.039179482856477606, -0.9368653687974748], [-0.3413907263607456, 0.8510267256748743, 0.45388728871916123, -0.8873483320810232, 2.3061526859499213, -1.1737848761313343], [0.5116470948621722, 0.5885168468728869, 1.0119527734469473, -0.8531670963007332, 0.32886167790602977, 0.01233855941669082]], [[-0.5409205894969416, 1.1302042956477851, 0.558862701468178, -0.04577449992525383, 1.4629958484374341, 0.7841447075459602], [0.7681347064675581, 0.7698865668464419, -0.056119575308986924, -0.07264464262556042, -0.039179482856477606, -0.9368653687974748], [-0.3413907263607456, 0.8510267256748743, 0.45388728871916123, -0.8873483320810232, 2.3061526859499213, -1.1737848761313343], [0.5116470948621722, 0.5885168468728869, 1.0119527734469473, -0.8531670963007332, 0.32886167790602977, 0.01233855941669082]], [[-0.5409205894969416, 1.1302042956477851, 0.558862701468178, -0.04577449992525383, 1.4629958484374341, 0.7841447075459602], [0.7681347064675581, 0.7698865668464419, -0.056119575308986924, -0.07264464262556042, -0.039179482856477606, -0.9368653687974748], [-0.3413907263607456, 0.8510267256748743, 0.45388728871916123, -0.8873483320810232, 2.3061526859499213, -1.1737848761313343], [0.5116470948621722, 0.5885168468728869, 1.0119527734469473, -0.8531670963007332, 0.32886167790602977, 0.01233855941669082]], [[-0.5409205894969416, 1.1302042956477851, 0.558862701468178, -0.04577449992525383, 1.4629958484374341, 0.7841447075459602], [0.7681347064675581, 0.7698865668464419, -0.056119575308986924, -0.07264464262556042, -0.039179482856477606, -0.9368653687974748], [-0.3413907263607456, 0.8510267256748743, 0.45388728871916123, -0.8873483320810232, 2.3061526859499213, -1.1737848761313343], [0.5116470948621722, 0.5885168468728869, 1.0119527734469473, -0.8531670963007332, 0.32886167790602977, 0.01233855941669082]]], "dtype": "float64"}, {"__nd__": [[0.0232800898697533, -0.06274468226561093, -0.10474678404731383, -0.06541698252396222, 0.0552322378268978, 0.1543961211402359], [0.0232800898697533, -0.06274468226561093, -0.10474678404731383, -0.06541698252396222, 0.0552322378268978, 0.1543961211402359], [0.0232800898697533, -0.06274468226561093, -0.10474678404731383, -0.06541698252396222, 0.0552322378268978, 0.1543961211402359], [0.0232800898697533, -0.06274468226561093, -0.10474678404731383, -0.06541698252396222, 0.0552322378268978, 0.1543961211402359], [0.0232800898697533, -0.06274468226561093, -0.10474678404731383, -0.06541698252396222, 0.0552322378268978, 0.1543961211402359]], "dtype": "float64"}, {"__nd__": [0.03313354176072773, 0.14088159849861487, 0.020258418923472826, 0.02355590420925225, 0.06472628692202095, -0.2825557503140886, 0.2313553612551031, 0.03512780330658696, -0.09129196545297188, -0.06767391091109551, 0.04653567649893719, -0.15405296469655982, -0.0035419462175941257, 0.0032490517579342737, -0.1441589841455158, -0.05645462800347718, 0.05729811774934569, 0.14360838885930716, 0.11095044289290525, 0.13686888586062262, 0.26530340992430457, -0.006249483657286997, -0.30388339611255577, -0.20298985890798976, -0.0232800898697533, 0.06274468226561093, 0.10474678404731383, 0.06541698252396222, -0.0552322378268978, -0.1543961211402359], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(dp_sgd_step, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")